# `0936` 앞발 들기: 0.5초와 0.25초 탐색 비교

**질문:** “강아지가 유리문 앞에서 앞발을 몇 번 들었다 내렸어? 뒷발이 바닥에 남아도 포함해.”

동일한 `gpt-4o`와 전·후 상태 검증 절차를 사용하고, **영상 전체 후보 탐색의 프레임 간격만** 0.5초에서 0.25초로 바꿨습니다. 아래 결과는 저장된 로컬 실험 JSON을 읽습니다. 표의 8개 사건 시각은 사용자 총횟수와 프레임 잠정 검토를 합친 기준이므로, 독립적인 완전한 정답 라벨로 간주하지 않습니다. 시간 매칭은 기준 구간 ±0.25초입니다.


## 핵심 결과

| 탐색 간격 | 탐색 프레임 | 후보 제안 | 8개 사건 중 후보 발견 | 전·후 검증 통과 | 최종 확정 |
|---|---:|---:|---:|---:|---:|
| 0.5초 | 64 | 6 | 6 | 5 | 3 |
| 0.25초 | 122 | 7 | 6 | 5 | 4 |

**두 탐색의 후보를 합치면 8개 사건 모두에 후보가 있습니다.** 0.25초가 단독으로 더 많은 사건을 찾은 것은 아닙니다. 초반 2개를 새로 찾는 대신 6~8초 사건 2개를 놓쳤습니다. 따라서 “프레임을 두 배 보내면 재현율이 오른다”고 결론 낼 수 없습니다. 새 0.25초 실행에는 모델 API 호출 25회가 발생했습니다(기존 캐시 적중 0회). 각 설정을 한 번씩만 실행했고 모델 응답 자체가 달라질 수 있으므로, 확정 횟수 차이를 프레임 간격만의 효과로 단정하지 않습니다.


## 사건별 탈락 단계

| 사람 기준 사건 | 구간(초) | 0.5초 탐색 | 0.25초 탐색 |
|---:|---:|---|---|
| 1 | 0.25~0.85 | 후보 없음 | 0.25초 후보 → 개체 근거 미확정 |
| 2 | 1.50~2.25 | 후보 없음 | 1.75초 후보 → 확정 |
| 3 | 4.25~4.90 | 4.75초 → 확정 | 4.50초 → 확정 |
| 4 | 6.00~6.75 | 6.25초 후보 → 전·후 검증 거절 | 후보 없음 |
| 5 | 7.40~8.00 | 7.75초 후보 → 이전 상태 복귀 근거 미확정 | 후보 없음 |
| 6 | 9.00~10.00 | 9.25초 후보 → 이전 상태 복귀 근거 미확정 | 9.25초 → 확정; 9.50초 추가 후보 거절 |
| 7 | 15.25~16.00 | 15.75초 → 확정 | 15.50초 → 확정 |
| 8 | 19.00~19.75 | 19.25초 → 확정 | 19.25초 후보 → 개체 연결 미확정 |

0.5초 실행에서 검증된 7.75초·9.25초 후보는 **같은 개체가 이전 상태로 돌아온 근거 부족** 때문에 최종 확정에서 빠졌습니다. 0.25초 실행의 19.25초 후보는 전·후 장면을 통과했지만 개체 ID가 `unknown`이라 빠졌습니다. 모델 판정의 근거를 그대로 보려면 아래 셀을 실행하세요.


In [1]:
import json
from pathlib import Path
root = Path.cwd()
if root.name == "notebooks": root = root.parent
report_path = root / "notebooks/smarthome-bench-evidence/new-pipeline/ablation-0936-step025/comparison.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
for step, metric in report["metrics"].items():
    print(f'{step}초: 탐색 프레임 {metric["scan_frames"]}, '
          f'후보가 있는 사건 {metric["reference_events_with_candidate"]}/8, '
          f'전·후 검증 통과 {metric["reference_events_verified_confirmed"]}/8, '
          f'최종 확정 {metric["reference_events_final_confirmed"]}/8')
print(f'두 탐색 결과의 후보 합집합: {report["union_candidate_coverage"]}/8 사건')


0.5초: 탐색 프레임 64, 후보가 있는 사건 6/8, 전·후 검증 통과 5/8, 최종 확정 3/8
0.25초: 탐색 프레임 122, 후보가 있는 사건 6/8, 전·후 검증 통과 5/8, 최종 확정 4/8
두 탐색 결과의 후보 합집합: 8/8 사건


In [ ]:
for event in report["events"]:
    print(f'\n사건 {event["event_number"]}: {event["reference_start"]}~{event["reference_end"]}초')
    for step in ("0.5", "0.25"):
        run = event["runs"][step]
        print(f'  {step}초: {run["stage"]}, 후보 {run["candidate_times"]}')
        for detail in run["details"]:
            print('    ', detail["time"], '검증', detail["verification_status"],
                  '최종', detail["final_decision"], '원인', detail["review_reason"] or detail["evidence_check_reason"])


## 원본 영상 확인

아래 셀은 로컬의 `data/videos/smartbench_0936.mp4`를 재생합니다. 저장된 비교 JSON과 표를 확인하는 데 영상 파일이나 API 키는 필요하지 않습니다. 사건 시각을 다시 검토할 때만 영상 파일이 필요합니다.


In [ ]:
video = root / "data/videos/smartbench_0936.mp4"
if not video.exists():
    print("로컬 영상 없음:", video)
else:
    try:
        from IPython.display import Video, display
    except ImportError:
        print("영상 재생은 Jupyter/IPython에서 열어주세요:", video)
    else:
        display(Video(filename=str(video), embed=False, html_attributes="controls"))


## 재실행 방법

프로젝트 루트에 원본 영상 `data/videos/smartbench_0936.mp4`와 OpenAI API 키가 필요합니다. 아래 첫 명령은 API 비용이 발생하고, 둘째 명령은 저장된 JSON만 비교합니다.

```bash
.venv/bin/python scripts/validate_video_aggregate.py --ids 0936 --scan-model gpt-4o --verify-model gpt-4o --coarse-step 0.25 --output notebooks/smarthome-bench-evidence/new-pipeline/ablation-0936-step025 --cache-dir notebooks/smarthome-bench-evidence/new-pipeline/api-cache
python3 scripts/compare_0936_sampling.py
```

기존 0.5초 결과는 `new-pipeline/smartbench_0936-result.json`, 새 0.25초 결과는 `new-pipeline/ablation-0936-step025/smartbench_0936-result.json`에 보존합니다.


## 다음 실험

1. **후보 합집합 검증:** 두 간격의 후보를 합치고 같은 사건에 속한 후보를 하나로 묶은 뒤, 8개 사건의 전·후·리셋 근거를 다시 검사합니다. 후보를 합쳤을 때의 최종 횟수는 아직 측정하지 않았습니다.
2. **경계 사건 재검토:** 0.25초의 첫 사건은 영상 시작 경계에서 개체 근거가 약했습니다. 6.25초 검증 거절은 ‘앞발 들기’ 상태 해석과 실제 프레임을 사람이 함께 확인해야 합니다.
3. **개체별 상태 흐름:** 같은 강아지의 `앞발 내림 → 들림 → 내림`을 시간순으로 기록하면 후보별로 리셋을 따로 추측하는 횟수를 줄일 수 있습니다. 후보 재현율·최종 확정률·미확정률을 분리해 평가합니다.
